# Trade Analysis Solution (Imports/Exports)

Worked solution aligned to `trade_analysis_exercise.ipynb`.


In [ ]:
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.colors import TwoSlopeNorm


## 0) Load data and standardize columns


In [ ]:
DATA_DIR = '../data/0_raw'
trade_data_path = os.path.join(DATA_DIR, 'Trade Data Python Training.xlsx')
df = pd.read_excel(trade_data_path)

# requested standardization right after loading
df.columns = df.columns.str.lower()

df.head()


## 1) Cleaning and replacements


In [ ]:
d = df.copy()

d['partner'] = d['partner'].astype(str).str.strip().str.upper()
d['flow'] = d['flow'].astype(str).str.strip().str.upper()

d['hscode'] = d['hscode'].astype('Int64').astype(str).str.replace('<NA>', '', regex=False).str.zfill(8)

if 'hscode2' not in d.columns:
    d['hscode2'] = pd.to_numeric(d['hscode'].str[:2], errors='coerce')
else:
    d['hscode2'] = pd.to_numeric(d['hscode2'], errors='coerce')

for col in ['year', 'period', 'mwkvalue', 'usdvalue']:
    if col in d.columns:
        d[col] = pd.to_numeric(d[col], errors='coerce')

d[['year', 'period', 'flow', 'partner', 'hscode', 'hscode2', 'mwkvalue', 'usdvalue']].head()


## 1.1) Create `usdvalue` from FX table (if missing)

If `usdvalue` is not available, create it by merging an FX table on `year` + `period`,
then compute `usdvalue = mwkvalue / mwk_per_usd`.


In [ ]:
# Build usdvalue from FX table only when missing
if 'usdvalue' not in d.columns or d['usdvalue'].isna().all():
    fx = pd.DataFrame({
        'year': [2024] * 12,
        'period': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
        'mwk_per_usd': [1700, 1710, 1720, 1730, 1740,
                        1750, 1760, 1770, 1780, 1790, 1800, 1810]
    })

    d = d.merge(fx, on=['year', 'period'], how='left', validate='m:1')
    d['usdvalue'] = d['mwkvalue'] / d['mwk_per_usd']
    d = d.drop(columns=['mwk_per_usd'])

# Ensure numeric dtype either way
d['usdvalue'] = pd.to_numeric(d['usdvalue'], errors='coerce')

d[['year', 'period', 'mwkvalue', 'usdvalue']].head()


## 2) Trade-Value Features and Outlier Handling


In [ ]:
sign_map = {'I': -1, 'R': -1, 'E': 1, 'RE': 1}
d['flow_sign'] = d['flow'].map(sign_map)

d['usdvalue_clean'] = pd.to_numeric(d['usdvalue'], errors='coerce')
if d['usdvalue_clean'].notna().sum() == 0:
    d['usdvalue_clean'] = d['mwkvalue'] / 1800

labels5 = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
notna_usd = d['usdvalue_clean'].dropna()
ranked = notna_usd.rank(method='first')
banded = pd.qcut(ranked, q=5, labels=labels5)
d['value_band'] = pd.Series(banded.values, index=notna_usd.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].boxplot(d['usdvalue_clean'].dropna(), vert=False)
axes[0].set_title('Before outlier filter')
axes[0].set_xlabel('usdvalue_clean')

q1 = d['usdvalue_clean'].quantile(0.25)
q3 = d['usdvalue_clean'].quantile(0.75)
iqr = q3 - q1
lower = max(0, q1 - 1.5 * iqr)
upper = q3 + 1.5 * iqr

d_no_outliers = d[d['usdvalue_clean'].between(lower, upper, inclusive='both')].copy()

axes[1].boxplot(d_no_outliers['usdvalue_clean'].dropna(), vert=False)
axes[1].set_title('After outlier filter')
axes[1].set_xlabel('usdvalue_clean')
plt.tight_layout()

print('Rows before:', len(d))
print('Rows after outlier filter:', len(d_no_outliers))


## 3) Create a date field


In [ ]:
d_work = d_no_outliers.copy()

d_work['date'] = pd.to_datetime(
    d_work['year'].astype('Int64').astype(str)
    + '-'
    + d_work['period'].astype('Int64').astype(str).str.zfill(2)
    + '-01',
    errors='coerce'
)
d_work['month_name'] = d_work['date'].dt.month_name()

d_work[['year', 'period', 'date', 'month_name']].head()


## 4) Seasonality from month name


In [ ]:
d_work['flow_group'] = np.where(d_work['flow_sign'] == -1, 'Import/Reimport', 'Export/Reexport')

month_order = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]
d_work['month_name'] = pd.Categorical(d_work['month_name'], categories=month_order, ordered=True)

seasonality = (
    d_work.groupby(['month_name', 'flow_group'], dropna=False)['mwkvalue']
    .sum()
    .reset_index(name='total_mwk')
    .sort_values('month_name')
)

fig, ax = plt.subplots(figsize=(10, 4))
for grp, g in seasonality.groupby('flow_group'):
    ax.plot(g['month_name'], g['total_mwk'], marker='o', label=grp)
ax.set_title('Seasonality by month name and flow group')
ax.set_xlabel('Month')
ax.set_ylabel('Total MWK')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()

seasonality.head()


## 5) New aggregation analyses


In [ ]:
yearly_flow_summary = (
    d_work.groupby(['year', 'flow_sign'], dropna=False)
    .agg(
        total_mwk=('mwkvalue', 'sum'),
        n_shipments=('mwkvalue', 'size'),
        avg_shipment_mwk=('mwkvalue', 'mean'),
        median_shipment_mwk=('mwkvalue', 'median')
    )
    .reset_index()
)

import_hs_mix = (
    d_work[d_work['flow_sign'] == -1]
    .groupby(['year', 'hscode2'], dropna=False)['mwkvalue']
    .sum()
    .reset_index(name='import_mwk')
)
import_hs_mix['share_of_imports_pct'] = (
    import_hs_mix['import_mwk']
    / import_hs_mix.groupby('year')['import_mwk'].transform('sum')
    * 100
)

yearly_flow_summary.head(), import_hs_mix.head()


## 6) Join + scatter visualization


In [ ]:
partner_imports = (
    d[d['flow'].isin(['I', 'R'])]
    .groupby('partner', dropna=False)
    .agg(imports_mwk=('mwkvalue', 'sum'), imports_usd=('usdvalue_clean', 'sum'))
    .reset_index()
)

partner_exports = (
    d[d['flow'].isin(['E', 'RE'])]
    .groupby('partner', dropna=False)
    .agg(exports_mwk=('mwkvalue', 'sum'), exports_usd=('usdvalue_clean', 'sum'))
    .reset_index()
)

partner_trade = partner_exports.merge(partner_imports, on='partner', how='outer').fillna(0)
partner_trade['balance_mwk'] = partner_trade['exports_mwk'] - partner_trade['imports_mwk']
partner_trade['balance_usd'] = partner_trade['exports_usd'] - partner_trade['imports_usd']
partner_trade['balance_sign'] = np.where(partner_trade['balance_mwk'] >= 0, 'Surplus', 'Deficit')

x = partner_trade['imports_mwk'].clip(lower=1)
y = partner_trade['exports_mwk'].clip(lower=1)

fig, ax = plt.subplots(figsize=(8, 6))
colors = partner_trade['balance_sign'].map({'Surplus': '#1f77b4', 'Deficit': '#d62728'})
ax.scatter(x, y, c=colors, alpha=0.7)

max_axis = max(x.max(), y.max())
ax.plot([1, max_axis], [1, max_axis], '--', color='gray', linewidth=1)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Imports (MWK, log scale)')
ax.set_ylabel('Exports (MWK, log scale)')
ax.set_title('Partner trade profile: imports vs exports')
plt.tight_layout()

partner_trade.head()


## 7.1) Geopandas step 1: download/read world country shapes


In [ ]:
world_url = 'https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip'
world_dir = Path(DATA_DIR).parent / 'external'
world_zip = world_dir / 'ne_110m_admin_0_countries.zip'

world_dir.mkdir(parents=True, exist_ok=True)
if not world_zip.exists():
    urllib.request.urlretrieve(world_url, world_zip)

world = gpd.read_file(world_zip)
world = world[['ADMIN', 'geometry']].copy()
world.head()


## 7.2) Geopandas step 2: prepare country-level trade balance


In [ ]:
country_balance = partner_trade[['partner', 'imports_mwk', 'exports_mwk', 'balance_mwk']].copy()
country_balance.head()


## 7.3) Geopandas step 3: harmonize names and merge


In [ ]:
world['partner'] = world['ADMIN'].str.upper().str.strip()
country_balance['partner'] = country_balance['partner'].str.upper().str.strip()

focus = pd.concat([
    country_balance.nlargest(10, 'balance_mwk'),
    country_balance.nsmallest(10, 'balance_mwk')
], axis=0).drop_duplicates(subset=['partner'])
focus_partners = set(focus['partner'])

name_fix_candidates = {
    'USA': 'UNITED STATES OF AMERICA',
    'UAE': 'UNITED ARAB EMIRATES',
    'UK': 'UNITED KINGDOM',
    'RUSSIA': 'RUSSIAN FEDERATION',
    'TANZANIA': 'UNITED REPUBLIC OF TANZANIA',
    'DR CONGO': 'DEMOCRATIC REPUBLIC OF THE CONGO'
}
name_fix = {k: v for k, v in name_fix_candidates.items() if k in focus_partners}

country_balance['partner_for_merge'] = country_balance['partner'].replace(name_fix)

geo_balance = world.merge(
    country_balance,
    left_on='partner',
    right_on='partner_for_merge',
    how='left'
)

focus_after_fix = focus.copy()
focus_after_fix['partner_for_merge'] = focus_after_fix['partner'].replace(name_fix)
unmatched_focus = sorted(set(focus_after_fix['partner_for_merge']) - set(world['partner']))
print('Unmatched focus countries (top/bottom 10):', unmatched_focus)


## 7.4) Geopandas step 4: map trade balance


In [ ]:
geo_balance['balance_mwk'] = geo_balance['balance_mwk'].fillna(0)

absmax = geo_balance['balance_mwk'].abs().max()
norm = TwoSlopeNorm(vmin=-absmax, vcenter=0, vmax=absmax)

ax = geo_balance.plot(
    column='balance_mwk',
    cmap='RdBu',
    norm=norm,
    figsize=(14, 7),
    edgecolor='white',
    linewidth=0.2,
    legend=True,
    missing_kwds={'color': 'lightgrey', 'label': 'No data'}
)
ax.set_title('Trade balance by partner country (MWK)')
ax.set_axis_off()
plt.tight_layout()


## Final checks


In [ ]:
print('All lowercase columns:', all(c == c.lower() for c in d.columns))
print('date dtype:', d_work['date'].dtype)
print('month_name exists:', 'month_name' in d_work.columns)
print('value_band levels:', sorted(pd.Series(d['value_band'].dropna().unique()).astype(str).tolist()))
